In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gauravchhajed/idid-dataset/ReadMe -V1.2 for IDID (1).pdf
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/labels_v1.2.json
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/3002017949 -V1.2 for IDID.pdf
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/170311 (2)v.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/160673h.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/160731v.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/100288h.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/140064.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/150454.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/161516v.JPG
/kaggle/input/datasets/gauravchhajed/idid-dataset/Train_IDID_V1.2/Train/Images/100023d.JPG

In [2]:
# Cell 1: Install dependencies
!pip install -q ultralytics

import os
import torch
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer

print("Ultralytics installed and imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics installed and imported successfully.


In [3]:
# Cell 1.5: Recreate YOLO Dataset from Raw IDID Data
import json, os, cv2, shutil, random, uuid
from tqdm import tqdm

# Setup paths from your Kaggle input
BASE_PATH = "/kaggle/input/datasets/gauravchhajed/idid-dataset"
IMG_PATH = BASE_PATH + "/Train_IDID_V1.2/Train/Images"
JSON_PATH = BASE_PATH + "/Train_IDID_V1.2/Train/labels_v1.2.json"
OUTPUT_PATH = "/kaggle/working/yolo_dataset"

# Create directories
os.makedirs(f"{OUTPUT_PATH}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/labels/train", exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/images/val", exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/labels/val", exist_ok=True)

# Class ID mapping
def get_class_id(obj):
    if "conditions" not in obj: return None
    values = list(obj["conditions"].values())
    if any("No issues" in v for v in values): return 0
    if any("Broken" in v for v in values): return 1
    if any("Flashover" in v for v in values): return 2
    return None

# 1. Parse JSON, format to YOLO txt, and copy images
print("Converting JSON to YOLO format...")
with open(JSON_PATH, 'r') as f: 
    data = json.load(f)

for item in tqdm(data):
    filename = item["filename"]
    img_file = os.path.join(IMG_PATH, filename)
    if not os.path.exists(img_file): continue
    
    img = cv2.imread(img_file)
    if img is None: continue
    h, w, _ = img.shape
    label_lines = []

    for obj in item["Labels"]["objects"]:
        if obj.get("string", 0) == 1: continue # Skip insulator strings
        class_id = get_class_id(obj)
        if class_id is None: continue
        
        x, y, bw, bh = obj["bbox"]
        x, y = max(0, x), max(0, y)
        bw, bh = min(bw, w - x), min(bh, h - y)
        
        x_center = (x + bw / 2) / w
        y_center = (y + bh / 2) / h
        label_lines.append(f"{class_id} {x_center} {y_center} {bw/w} {bh/h}")

    if label_lines:
        label_file = os.path.join(OUTPUT_PATH, "labels/train", os.path.splitext(filename)[0] + ".txt")
        with open(label_file, "w") as f: f.write("\n".join(label_lines))
        shutil.copy(img_file, os.path.join(OUTPUT_PATH, "images/train"))

# 2. Train/Val Split (80/20)
print("Splitting into Train and Val sets...")
random.seed(42)
images = os.listdir(f"{OUTPUT_PATH}/images/train")
random.shuffle(images)
val_imgs = images[int(0.8 * len(images)):]

for img in val_imgs:
    label = os.path.splitext(img)[0] + ".txt"
    shutil.move(os.path.join(OUTPUT_PATH, f"images/train/{img}"), os.path.join(OUTPUT_PATH, f"images/val/{img}"))
    if os.path.exists(os.path.join(OUTPUT_PATH, f"labels/train/{label}")):
        shutil.move(os.path.join(OUTPUT_PATH, f"labels/train/{label}"), os.path.join(OUTPUT_PATH, f"labels/val/{label}"))

# 3. Duplicate minority class (Broken)
print("Augmenting minority class (Broken)...")
labels_path = f"{OUTPUT_PATH}/labels/train"
images_path = f"{OUTPUT_PATH}/images/train"
image_map = {os.path.splitext(img)[0]: img for img in os.listdir(images_path)}

for file in os.listdir(labels_path):
    with open(os.path.join(labels_path, file)) as f: lines = f.readlines()
    
    if any(line.startswith("1 ") for line in lines): # If contains Broken class
        base_name = os.path.splitext(file)[0]
        if base_name in image_map:
            img_name = image_map[base_name]
            for _ in range(2): # Duplicate twice
                uid = str(uuid.uuid4())[:6]
                shutil.copy(os.path.join(images_path, img_name), os.path.join(images_path, f"copy_{uid}_{img_name}"))
                shutil.copy(os.path.join(labels_path, file), os.path.join(labels_path, f"copy_{uid}_{file}"))

print("Data prep complete! The yolo_dataset is ready in /kaggle/working/")

Converting JSON to YOLO format...


100%|██████████| 1600/1600 [02:37<00:00, 10.17it/s]


Splitting into Train and Val sets...
Augmenting minority class (Broken)...
Data prep complete! The yolo_dataset is ready in /kaggle/working/


In [4]:
# Cell 2: Fix data.yaml and save to working directory
import yaml
import shutil

original_yaml_path = "/kaggle/input/datasets/gauravchhajed/dataset-1/data (1).yaml"
working_yaml_path = "/kaggle/working/data.yaml"

if os.path.exists(original_yaml_path):
    with open(original_yaml_path, 'r') as file:
        data = yaml.safe_load(file)
    
    # Update paths to point to your preprocessed YOLO dataset directory
    # (Adjust these paths if your preprocessed images are saved elsewhere)
    data['path'] = "/kaggle/working/yolo_dataset" 
    data['train'] = "images/train"
    data['val'] = "images/val"
    
    with open(working_yaml_path, 'w') as file:
        yaml.dump(data, file, default_flow_style=False)
        
    print(f"Updated YAML saved to {working_yaml_path}")
    print(data)
else:
    print(f"Error: Could not find {original_yaml_path}")

Updated YAML saved to /kaggle/working/data.yaml
{'path': '/kaggle/working/yolo_dataset', 'train': 'images/train', 'val': 'images/val', 'names': {0: 'good', 1: 'broken', 2: 'flashover'}}


In [5]:
# Cell 3: Custom KD Trainer Class (CORRECTED)
class KDDetectionTrainer(DetectionTrainer):
    # Use *args and **kwargs to let Ultralytics handle its own default arguments safely
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        self.kd_alpha = 0.3 
        
        # Updated with your specific Kaggle dataset path
        teacher_path = "/kaggle/input/datasets/gauravchhajed/teacher-yolov8m-weights/final_best_model_v3_082.pt"
        
        if not os.path.exists(teacher_path):
            raise FileNotFoundError(f"Teacher model missing! Expected it at: {teacher_path}")
            
        print(f"Loading Teacher Model from {teacher_path}...")
        self.teacher = YOLO(teacher_path).model.to(self.device)
        
        self.teacher.eval()
        for param in self.teacher.parameters():
            param.requires_grad = False

    def train_step(self, batch):
        """Custom training step incorporating MSE Distillation Loss."""
        batch = self.preprocess_batch(batch)
        self.optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(self.amp):
            # --- STUDENT FORWARD PASS ---
            student_preds = self.model(batch["img"])
            base_loss, loss_items = self.criterion(student_preds, batch)
            
            # --- TEACHER FORWARD PASS ---
            with torch.no_grad():
                teacher_preds = self.teacher(batch["img"])
                
            # --- DISTILLATION LOSS (Feature Map MSE) ---
            kd_loss = 0.0
            if isinstance(student_preds, (list, tuple)):
                for sp, tp in zip(student_preds, teacher_preds):
                    kd_loss += F.mse_loss(sp, tp)
            else:
                kd_loss = F.mse_loss(student_preds, teacher_preds)
            
            # --- COMBINE LOSSES ---
            total_loss = base_loss + (self.kd_alpha * kd_loss)
            
        # Backward pass & Optimizer step
        self.scaler.scale(total_loss).backward()
        self.scaler.unscale_(self.optimizer)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        
        # Update loss array for the console/logger
        loss_items[0] = total_loss.detach() 
        
        return loss_items

In [6]:
# Cell 4: Start KD Training
args = dict(
    model="yolov8n.pt",                 
    data="/kaggle/working/data.yaml",   # Pointing to the fixed YAML
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,                           
    augment=True,                       
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    project="/kaggle/working/kd_project", 
    name="yolov8n_distilled"
)

print("Starting Knowledge Distillation Training...")
trainer = KDDetectionTrainer(overrides=args)
trainer.train()

Starting Knowledge Distillation Training...
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_distilled, nbs=64, nms=False, opset=None, optimize=False, optimizer=au